In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git"
STUDENT_ROLE = "IT24100533 - Danthanarayana D.M.R"
STUDENT_NAME = input("Enter your real GitHub display name: ").strip()
STUDENT_EMAIL = input("Enter your verified GitHub email: ").strip()
CLONE_DIR = Path("/content/IT3091---Machine-Learning-Assignment")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
os.chdir(CLONE_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run(["git", "config", "user.name", STUDENT_NAME], check=True)
subprocess.run(["git", "config", "user.email", STUDENT_EMAIL], check=True)
print("Ready for", STUDENT_ROLE)
print("Authenticate your own GitHub account before running the final cell.")

# Weather feature preparation

This notebook joins the cleaned production table to locations and creates weather summaries from the three months before each growing-season forecast date.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
production = pd.read_csv(ROOT / 'data' / 'preprocessed' / 'production_clean.csv')
locations = pd.read_csv(ROOT / 'data' / 'raw' / 'locationData.csv')
weather = pd.read_csv(ROOT / 'data' / 'raw' / 'weatherData.csv')
weather['date'] = pd.to_datetime(weather['date'], errors='coerce')
production = production[production['District'].isin(locations['city_name'])].copy()
weather = weather.merge(locations[['location_id', 'city_name']], on='location_id', how='inner').rename(columns={'city_name': 'District'})
column_map = {}
for prefix, target in [('rain_sum', 'Rainfall_mm'), ('temperature_2m_mean', 'Temperature_C'), ('wind_speed_10m_max', 'WindSpeed_kmh'), ('shortwave_radiation_sum', 'SolarRadiation_MJ_m2'), ('et0_fao_evapotranspiration', 'ET0_mm')]:
    matches = [c for c in weather.columns if str(c).startswith(prefix)]
    if matches:
        column_map[matches[0]] = target
weather = weather.rename(columns=column_map)
features = []
for district, year, season in production[['District', 'Year', 'Season']].drop_duplicates().itertuples(index=False, name=None):
    anchor = pd.Timestamp(year=int(year), month=4 if season == 'Yala' else 9, day=1)
    window = weather[(weather['District'] == district) & (weather['date'] < anchor) & (weather['date'] >= anchor - pd.DateOffset(months=3))]
    if len(window) < 80:
        continue
    features.append({'District': district, 'Year': year, 'Season': season, 'ForecastDate': anchor, 'Rainfall_3M_mm': window['Rainfall_mm'].sum(), 'TempMean_3M_C': window['Temperature_C'].mean(), 'WindSpeed_3M_kmh': window['WindSpeed_kmh'].mean(), 'SolarRadiation_3M_MJ_m2': window['SolarRadiation_MJ_m2'].mean(), 'ET0_3M_mm': window['ET0_mm'].sum(), 'WeatherDays_3M': len(window)})
result = production.merge(pd.DataFrame(features), on=['District', 'Year', 'Season'], how='inner')
result.to_csv(ROOT / 'data' / 'preprocessed' / 'crop_yield_preprocessed.csv', index=False)
print('Weather rows:', len(weather))
print('Yield rows with weather features:', len(result))
print('Saved crop_yield_preprocessed.csv')

Only weather information available before the forecast date is used. The realised yield is kept as the prediction target.

In [ ]:
import subprocess

print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout)
subprocess.run(["git", "add", "notebooks/02_weather_data_danthanarayana.ipynb"], check=True)
subprocess.run(["git", "add", "data/preprocessed/crop_yield_preprocessed.csv"], check=True)
subprocess.run(["git", "commit", "-m", "Add weather features to yield data"], check=False)
subprocess.run(["git", "push", "origin", "main"], check=True)
print("Contribution pushed to main.")